# Quy trình GPU trên ckey.vn: tìm card ≥ 8GB VRAM rẻ nhất → thuê → cấu hình dự án

Notebook chạy tuần tự từ trên xuống:

1. Kiểm tra số dư tài khoản
2. Tìm GPU 1 card, VRAM 8–12 GB, giá rẻ nhất
3. **Thuê GPU** (tốn tiền, có bước xác nhận) bằng template *Open WebUI + Ollama* (id 12)
4. Chờ máy chạy và lấy địa chỉ Ollama (host + port public)
5. Pull model LLM lên server từ xa
6. Ghi `.env` (`RAG_PROFILE=server`, `OLLAMA_BASE_URL`, `CKEY_GPU_INSTANCE_ID`)
7. (Tuỳ chọn) tải model embedding/reranker về máy bạn
8. Kiểm tra kết nối end-to-end
9. Xoá GPU khi xong, để không bị tính tiền tiếp

**Cần trong `.env`:** `CKEY_API_KEY`. Mật khẩu máy thuê lấy từ `CKEY_GPU_PASSWORD` hoặc được hỏi lúc chạy.

> Lưu ý: GPU thuê **chỉ chạy LLM** (Ollama). Embedding `AITeamVN/Vietnamese_Embedding` (vector 1024 chiều) và reranker `AITeamVN/Vietnamese_Reranker` chạy **ở máy bạn**, nạp ở fp16 (`use_fp16=True` trong `CONFIG_SERVER`) để cả hai vừa GPU 4 GB (~2,3 GB VRAM). Ô cuối notebook kiểm tra việc này.

In [1]:
import getpass
import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Any

import requests
from dotenv import load_dotenv, set_key

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
ENV_FILE = ROOT / ".env"
load_dotenv(ENV_FILE)

from scripts.manager import ServiceManager
from src.config import CONFIG_SERVER

# ---- Tham số có thể chỉnh ----
MIN_VRAM_GB = 8              # VRAM tối thiểu
MAX_VRAM_GB = 12             # không lấy card quá lớn (đắt không cần thiết)
MAX_PRICE_VND_PER_HOUR = 3000
TEMPLATE_ID = "12"           # 12 = Open WebUI + Ollama (cần port 22, 8080, 11434)
PORTS = "22,8080,11434"
OLLAMA_PORT = 11434
LLM_MODEL = CONFIG_SERVER.llm_model_name   # qwen3:8b, đồng bộ với src/config.py
WAIT_TIMEOUT_S = 900
DOWNLOAD_LOCAL_HF_MODELS = False  # True: tải embedding/reranker về máy ở bước 7

mgr = ServiceManager()
print("ROOT:", ROOT, "| LLM:", LLM_MODEL)

ROOT: /mnt/data/1_Projects/sgu-rag-llamaindex | LLM: qwen3:8b


## 1. Số dư tài khoản

In [2]:
print(json.dumps(mgr.profile(), indent=2, ensure_ascii=False))

{
  "success": true,
  "status": 200,
  "message": "Lấy thông tin profile thành công",
  "data": {
    "profile": {
      "username": "huynhthanhtien830",
      "name": "Huỳnh Thanh Tiến",
      "email": "huynhthanhtien830@gmail.com",
      "balance": "9.263 VND",
      "balance_raw": 9263.184113,
      "created_at": "29/09/2026 - 13:46:03",
      "created_at_timestamp": 1790664363,
      "api_key_masked": "b6ab****************************************2b00"
    }
  }
}


## 2. Tìm GPU rẻ nhất có VRAM 8–12 GB

`gpu_name` có dạng `1x GTX 1660 Ti - 6 GB`, nên notebook tách số card và VRAM bằng regex rồi lọc.

In [3]:
def parse_gpu(g: dict[str, Any]) -> dict[str, Any] | None:
    name = str(g.get("gpu_name", ""))
    vram = re.search(r"(\d+(?:\.\d+)?)\s*GB\s*$", name)
    cards = re.match(r"\s*(\d+)x", name)
    price = g.get("raw_price")
    if not vram or not cards or not isinstance(price, (int, float)):
        return None
    return {**g, "vram_gb": float(vram.group(1)), "cards": int(cards.group(1)), "price_h": float(price)}

raw = mgr.find_gpus(gpu="all", sort="price_asc", count="1")
parsed = [p for g in raw if (p := parse_gpu(g))]
candidates = sorted(
    (
        p for p in parsed
        if p["cards"] == 1
        and MIN_VRAM_GB <= p["vram_gb"] <= MAX_VRAM_GB
        and p["price_h"] <= MAX_PRICE_VND_PER_HOUR
    ),
    key=lambda p: p["price_h"],
)
print(f"{len(raw)} node trả về, {len(candidates)} node thoả điều kiện\n")
for p in candidates[:10]:
    print(f"id={p['id']:>8} | {p['gpu_name']:<32} | {p['price_h']:>7.0f} VND/h | RAM {p.get('ram')} | CPU {p.get('cpu')}")

if not candidates:
    raise RuntimeError("Không có GPU nào thoả điều kiện — nới MAX_PRICE_VND_PER_HOUR hoặc khoảng VRAM.")
chosen = candidates[0]
print("\n=> Chọn:", chosen["id"], chosen["gpu_name"], f"{chosen['price_h']:.0f} VND/h")

567 node trả về, 17 node thoả điều kiện

id=  120186 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 32 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120187 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 32 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120188 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 32 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120189 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 32 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120191 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 24 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120193 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 32 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120196 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 24 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120197 | 1x Tesla P4 - 8 GB               |     853 VND/h | RAM 32 GB | CPU Intel(R) Xeon(R) CPU @ 2.00GHz
id=  120201 | 1x Tesla P4 - 8 GB               |     853 VND/h 

## 3. Thuê GPU — **tốn tiền**

Chạy ô này sẽ hỏi xác nhận (`y`). Mật khẩu lấy từ `CKEY_GPU_PASSWORD` hoặc được hỏi.

In [4]:
password = os.environ.get("CKEY_GPU_PASSWORD") or getpass.getpass("Mật khẩu cho máy thuê: ")

print(f"Sắp thuê node {chosen['id']} ({chosen['gpu_name']}) ~{chosen['price_h']:.0f} VND/giờ")
print(f"Template {TEMPLATE_ID}, ports {PORTS}")
if input("Tiếp tục? [y/N] ").strip().lower() != "y":
    raise RuntimeError("Đã huỷ, chưa thuê gì.")

rent = mgr.rent_gpu(str(chosen["id"]), TEMPLATE_ID, password, PORTS)
print(json.dumps(rent, indent=2, ensure_ascii=False)[:1500])

rent_id = rent.get("id") if isinstance(rent, dict) else None
if not rent_id:
    raise RuntimeError("Phản hồi không có 'id' — kiểm tra output ở trên rồi đặt instance_id thủ công.")
instance_id: str = str(rent_id)
set_key(str(ENV_FILE), "CKEY_GPU_INSTANCE_ID", instance_id)
print("Instance ID:", instance_id, "(đã lưu vào .env)")

Sắp thuê node 120186 (1x Tesla P4 - 8 GB) ~853 VND/giờ
Template 12, ports 22,8080,11434
{
  "status": 200,
  "message": "Thuê GPU thành công!",
  "id": "120186"
}
Instance ID: 120186 (đã lưu vào .env)


## 4. Chờ máy chạy và lấy địa chỉ Ollama

`infogpu3` trả `public_ipaddr` và thông tin port. **Định dạng map port chưa được tài liệu mô tả**, nên hàm dưới dò theo nhiều dạng thường gặp. Nếu không tìm ra, ô này in thông tin thô; khi đó hãy đặt `OLLAMA_URL_OVERRIDE`, ví dụ `"http://n1.ckey.vn:34567"`.

In [5]:
OLLAMA_URL_OVERRIDE: str | None = None


def walk(node: Any, path: str = ""):
    yield path, node
    if isinstance(node, dict):
        for k, v in node.items():
            yield from walk(v, f"{path}/{k}")
    elif isinstance(node, list):
        for i, v in enumerate(node):
            yield from walk(v, f"{path}[{i}]")


def external_port(info: Any, internal: int) -> int | None:
    """Dò port public ứng với port nội bộ `internal` trong phản hồi infogpu3."""
    tag = str(internal)
    for path, node in walk(info):
        # Dạng {"11434/tcp": 34567} hoặc {"11434": "34567"}
        if re.search(rf"/{tag}(/tcp|/udp)?$", path) and isinstance(node, (int, str)):
            m = re.search(r"\d{4,5}", str(node))
            if m and int(m.group()) != internal:
                return int(m.group())
        # Dạng chuỗi "34567:11434", "11434->34567", "34567->11434/tcp"
        if isinstance(node, str) and tag in node:
            nums = [int(n) for n in re.findall(r"\d{4,5}", node) if int(n) != internal]
            if len(nums) == 1:
                return nums[0]
        # Dạng {"internal": 11434, "external": 34567}
        if isinstance(node, dict) and internal in [v for v in node.values() if isinstance(v, int)]:
            others = [v for v in node.values() if isinstance(v, int) and v != internal and v >= 1024]
            if len(others) == 1:
                return others[0]
    return None


def resolve_ollama_url(info: Any) -> str | None:
    if OLLAMA_URL_OVERRIDE:
        return OLLAMA_URL_OVERRIDE.rstrip("/")
    host = info.get("public_ipaddr") if isinstance(info, dict) else None
    port = external_port(info, OLLAMA_PORT)
    return f"http://{host}:{port}" if host and port else None


def ollama_up(url: str) -> bool:
    try:
        return requests.get(f"{url}/api/version", timeout=8).ok
    except requests.RequestException:
        return False


ollama_url: str | None = None
deadline = time.time() + WAIT_TIMEOUT_S
last_dump = ""
while time.time() < deadline:
    info = mgr.gpu_info(instance_id)
    dump = json.dumps(info, ensure_ascii=False, sort_keys=True)
    if dump != last_dump:
        print(time.strftime("%H:%M:%S"), "infogpu3:", dump[:800])
        last_dump = dump
    url = resolve_ollama_url(info)
    if url and ollama_up(url):
        ollama_url = url
        break
    time.sleep(15)

if ollama_url:
    print("\nOllama sẵn sàng:", ollama_url)
else:
    print("\nChưa lấy được địa chỉ Ollama. Xem infogpu3 ở trên, đặt OLLAMA_URL_OVERRIDE rồi chạy lại ô này.")

16:54:40 infogpu3: {"access_tag": "", "billed_total": "0 VND", "billed_total_raw": 0, "connection_label": "Web UI URL", "connection_type": "url", "connection_url": "http://n3.ckey.vn:2285", "connection_value": "http://n3.ckey.vn:2285", "cpu_name": "4/8", "desktop_port": "N/A", "geolocation": "US", "gpu_name": "1x Tesla P4 - 8 GB", "host_type_tag": "Community", "mon_container_status": {"class": "badge-warning", "text": "Đang khởi tạo"}, "mrl": "14-10-2026 04:54:35", "online_status": {"class": "badge-success", "text": "Hoạt động"}, "operating_system": "chieustudio/openwebui-ollama-ubuntu:latest", "port_forwards": {"11434": "2286", "22": "2284", "8080": "2285"}, "price": "853 VND", "public_host_label": "IPv4", "public_ipaddr": "n3.ckey.vn", "ram": "32", "ssh_command": "ssh root@n3.ckey.vn -p 2284", "ssh_port": 
16:56:16 infogpu3: {"access_tag": "", "billed_total": "21 VND", "billed_total_raw": 20.61646, "connection_label": "Web UI URL", "connection_type": "url", "connection_url": "http://

## 5. Pull model LLM lên server từ xa

In [6]:
assert ollama_url, "Chưa có ollama_url — chạy lại bước 4."

with requests.post(f"{ollama_url}/api/pull", json={"model": LLM_MODEL, "stream": True}, stream=True, timeout=None) as r:
    r.raise_for_status()
    last_status, last_pct = "", -1
    for line in r.iter_lines():
        if not line:
            continue
        ev = json.loads(line)
        if ev.get("error"):
            raise RuntimeError(ev["error"])
        status = ev.get("status", "")
        pct = int(ev["completed"] / ev["total"] * 10) * 10 if ev.get("total") and ev.get("completed") else -1
        if status != last_status or pct != last_pct:
            print(status + (f" {pct}%" if pct >= 0 else ""))
            last_status, last_pct = status, pct

tags = [m["name"] for m in requests.get(f"{ollama_url}/api/tags", timeout=15).json().get("models", [])]
print("Model trên server:", tags)

pulling manifest
pulling a3de86cd1c13
pulling a3de86cd1c13 0%
pulling a3de86cd1c13 10%
pulling a3de86cd1c13 20%
pulling a3de86cd1c13 30%
pulling a3de86cd1c13 40%
pulling a3de86cd1c13 50%
pulling a3de86cd1c13 60%
pulling a3de86cd1c13 70%
pulling a3de86cd1c13 80%
pulling a3de86cd1c13 90%
pulling a3de86cd1c13 100%
pulling ae370d884f10
pulling ae370d884f10 100%
pulling d18a5cc71b84
pulling d18a5cc71b84 100%
pulling cff3f395ef37
pulling cff3f395ef37 100%
pulling 05a61d37b084
pulling 05a61d37b084 100%
verifying sha256 digest
writing manifest
success
Model trên server: ['qwen3:8b']


## 6. Ghi cấu hình vào `.env`

Đặt `RAG_PROFILE=server` và `OLLAMA_BASE_URL` trỏ tới GPU vừa thuê. Các dòng khác trong `.env` được giữ nguyên.

In [7]:
assert ollama_url, "Chưa có ollama_url — chạy lại bước 4."
set_key(str(ENV_FILE), "RAG_PROFILE", "server")
set_key(str(ENV_FILE), "OLLAMA_BASE_URL", ollama_url)
set_key(str(ENV_FILE), "CKEY_GPU_INSTANCE_ID", instance_id)
print(".env đã cập nhật: RAG_PROFILE=server, OLLAMA_BASE_URL=" + ollama_url)

.env đã cập nhật: RAG_PROFILE=server, OLLAMA_BASE_URL=http://n3.ckey.vn:2286


## 7. (Tuỳ chọn) Tải model embedding và reranker về máy

Đặt `DOWNLOAD_LOCAL_HF_MODELS = True` ở ô cấu hình đầu notebook. Dung lượng khoảng vài GB.

In [8]:
if DOWNLOAD_LOCAL_HF_MODELS:
    from huggingface_hub import snapshot_download

    for repo in (CONFIG_SERVER.embed_model_name, CONFIG_SERVER.reranker_model_name):
        if repo:
            print("Tải", repo)
            print("  ->", snapshot_download(repo_id=repo))
else:
    print("Bỏ qua (DOWNLOAD_LOCAL_HF_MODELS = False).")

Bỏ qua (DOWNLOAD_LOCAL_HF_MODELS = False).


## 8. Kiểm tra end-to-end

In [9]:
import importlib

os.environ["RAG_PROFILE"] = "server"
assert ollama_url, "Chưa có ollama_url — chạy lại bước 4."
os.environ["OLLAMA_BASE_URL"] = ollama_url
import src.config as cfg_module
importlib.reload(cfg_module)
cfg = cfg_module.get_config()
print("Config:", cfg.llm_model_name, "@", cfg.ollama_base_url)

t0 = time.time()
resp = requests.post(
    f"{cfg.ollama_base_url}/api/chat",
    json={"model": cfg.llm_model_name, "messages": [{"role": "user", "content": "Chào bạn, bạn là ai?"}], "stream": False},
    timeout=cfg.request_timeout,
)
resp.raise_for_status()
print(f"Trả lời sau {time.time() - t0:.1f}s:", resp.json()["message"]["content"][:300])

Config: qwen3:8b @ http://n3.ckey.vn:2286
Trả lời sau 15.0s: Chào bạn! Tôi là Qwen, một mô hình ngôn ngữ lớn được phát triển bởi Alibaba Cloud. Tôi có thể giúp bạn trả lời câu hỏi, tạo nội dung, và trò chuyện về nhiều chủ đề khác nhau. Bạn có gì cần hỏi không? 😊


## 9. Dọn dẹp — xoá GPU khi dùng xong

GPU thuê tính tiền theo giờ cho đến khi bị xoá. Đặt `DELETE_NOW = True` và chạy ô dưới (có hỏi xác nhận). Thao tác không hoàn tác được.

In [ ]:
DELETE_NOW = False

if DELETE_NOW:
    if input(f"XOÁ GPU instance {instance_id}? [y/N] ").strip().lower() == "y":
        print(mgr.gpu_action(instance_id, "delete"))
    else:
        print("Đã huỷ.")
else:
    print("DELETE_NOW = False — GPU vẫn đang chạy và đang tính tiền.")

{'message': 'Delete successful', 'success': True, 'already_deleted': False}


In [ ]:
import time

import torch
from llama_index.core import QueryBundle, Settings
from llama_index.core.schema import NodeWithScore, TextNode

from src.index_builder import setup_embedding
from src.retriever import get_postprocessors

# Nạp embedding + reranker trên máy bạn, đúng như pipeline dùng (fp16 khi cfg.use_fp16 và có cuda).
if cfg.embed_device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Máy không có CUDA — đổi embed_device sang 'cpu' trong src/config.py.")
t0 = time.time()
setup_embedding(cfg)
reranker = get_postprocessors(cfg)[0]
print(f"Nạp embedding + reranker sau {time.time() - t0:.1f}s (fp16={cfg.use_fp16})")

query = "Điều kiện để sinh viên được xét tốt nghiệp là gì?"
passages = [
    "Sinh viên được xét tốt nghiệp khi tích lũy đủ số tín chỉ và đạt chuẩn đầu ra theo chương trình đào tạo.",
    "Sinh viên được phép bảo lưu kết quả học tập tối đa hai học kỳ liên tiếp.",
    "Học phí được thu theo số tín chỉ đăng ký trong mỗi học kỳ.",
]

t0 = time.time()
q_vec = Settings.embed_model.get_query_embedding(query)
p_vecs = Settings.embed_model.get_text_embedding_batch(passages)
print(f"Embedding: {len(q_vec)} chiều, encode {len(passages) + 1} câu trong {time.time() - t0:.2f}s")
for vec, text in sorted(zip(p_vecs, passages), key=lambda x: -sum(a * b for a, b in zip(q_vec, x[0]))):
    print(f"  cos={sum(a * b for a, b in zip(q_vec, vec)):.4f} | {text}")

nodes = [NodeWithScore(node=TextNode(text=t), score=0.0) for t in passages]
ranked = reranker.postprocess_nodes(nodes, QueryBundle(query))
print("Rerank:")
for n in ranked:
    print(f"  {n.score:.4f} | {n.node.get_content()}")

if torch.cuda.is_available():
    print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1024**2:.0f} MB / "
          f"{torch.cuda.get_device_properties(0).total_memory / 1024**2:.0f} MB")